# PHASE: W9-G-PRE — Action-specific verification

**SCIENTIFIC QUESTION:** Is there enough saved-output headroom to learn whether THIS proposed F0 answer should replace D0?

**PREREQUISITES:** Complete immutable W9-F FIT outputs; frozen F2-A prompt; unchanged E3-k3 evidence.

**INPUT ARTIFACTS:** `configs/week9/w9g_action_verification.yaml`; saved D0/F0/F2-A answers/scores, F handoff/freezes and E3 identities.

**EXPECTED OUTPUTS:** new `results/week9/w9g_action_verification/v1/preflight` reports/decision; disabled G2 scaffold. Historical G is untouched.

**CURRENT SPLIT:** FIT only, no DEV/INTERNAL-VAL/TEST selection.

**FULL RUN FLAG:** `RUN_FULL=False`; `RUN_CONTROLLER_TRAINING=False`. `RUN_PREFLIGHT=True` performs cheap saved-output analysis with zero model generation. No query-to-K controller is used.

## 1. Protocol

Selection -> Fidelity -> Utilization -> Action Verification. Fixed evidence and already-generated answers define the action. Oracles are retrospective ceilings, not deployable controllers. F2-A is the frozen F2 variant; the separate overall W9-F intervention freeze may have selected F0. Preserve both artifacts and evaluate the three requested action spaces explicitly.


In [1]:
from pathlib import Path
import sys, json, subprocess
import pandas as pd
from IPython.display import display
ROOT = Path.cwd().resolve()
while not (ROOT/'src/week9').is_dir():
    if ROOT == ROOT.parent:
        raise FileNotFoundError('Open within CS787-RAG-Project')
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.week9.provenance import branch_guard
from src.week9.action_verification_g.protocol import config
branch_guard()
RUN_PREFLIGHT = True
RUN_CONTROLLER_TRAINING = True
RUN_FULL = False
SMOKE_LIMIT = 2
if RUN_FULL:
    raise ValueError('No full LM generation exists in this action-specific notebook')
CONFIG = ROOT/'configs/week9/w9g_action_verification.yaml'
cfg = config(CONFIG)
RESULT_ROOT = ROOT/cfg['output_root']
PREFLIGHT = RESULT_ROOT/'preflight'
def show_csv(name):
    path = PREFLIGHT/name
    if path.exists():
        display(pd.read_csv(path))
    else:
        print('Pending preflight:', name)
def read_json(path):
    return json.loads(path.read_text())


## 2. Load frozen W9-F handoff

The compatibility loader reads original F results as well as the handoff. This matters because the handoff contains the globally frozen intervention (F0 here), while PRE-A/C also need frozen F2-A. No original F file is rewritten.


In [2]:
froot = ROOT/cfg['f_root']
display({'frozen_F2_variant': read_json(froot/'frozen_w9f_f2.json')['selected_variant'],
         'overall_F_intervention': read_json(froot/'frozen_w9f_intervention.json')['actual_policy'],
         'evidence_policy': read_json(froot/'frozen_w9f_intervention.json')['evidence_policy']})


{'frozen_F2_variant': 'F2-A',
 'overall_F_intervention': 'F0',
 'evidence_policy': 'E3-k3'}

## 3. Alignment and hash assertions / execute saved-output preflight

Exact FIT query/paper coverage, complete F/E3 identities, saved row checksums, source sentence IDs/text hashes, original draft/score and F0 alias equivalence are mandatory. Missing/misaligned actions fail loudly. No tokenizer, generator or semantic encoder is loaded.


In [3]:
if RUN_PREFLIGHT:
    subprocess.run([str(ROOT/'sara_env/bin/python'), '-m', 'src.week9.action_verification_g.runner',
                    '--config', str(CONFIG), '--stage', 'preflight', '--resume'], cwd=ROOT, check=True)
else:
    print('Automatic preflight disabled; displaying existing saved artifacts only.')
if (PREFLIGHT/'preflight_manifest.json').exists():
    manifest = read_json(PREFLIGHT/'preflight_manifest.json')
    display({k:v for k,v in manifest.items() if k != 'artifact_hashes'})


Validating saved FIT F/E3 actions and hashes; no generator/encoder is loaded.
{
  "G1": {
    "decision": "NO-GO",
    "epsilon_headroom": 0.0038332437934255337,
    "threshold": 0.005,
    "rule": "epsilon oracle(D0,F2-A) minus best fixed >= threshold"
  },
  "G2": {
    "decision": "GO",
    "epsilon_headroom": 0.06381925695899743,
    "verifier_advantage": 0.059986013165571894,
    "threshold": 0.005,
    "material_advantage_threshold": 0.005,
    "rule": "epsilon oracle(D0,F0) headroom >=.005 AND advantage over PRE-A headroom >=.005; descriptive screen, not significance"
  },
  "G3": {
    "decision": "NO-GO",
    "extra_exact_oracle_f1": 0.0009634178579209784,
    "unique_best_rate": 0.0074455899198167235,
    "threshold": 0.003,
    "rule": "Three-action exact oracle minus D0/F0 exact oracle >=.003. Unique-best fraction is descriptive; no outcome-tuned extra cutoff."
  },
  "recommended_route": "G2",
  "controller_training_started": false,
  "lm_calls": 0,
  "interpretation": "Re

{'schema': 'W9-G-action-verification-v1',
 'split': 'fit',
 'complete': True,
 'queries': 1746,
 'papers': 656,
 'protocol_hash': '51aa9d5351ef5800991126439c21ac0a225a9f3db34d11258e31617a5b500503',
 'old_artifacts_unchanged': True,
 'lm_calls': 0,
 'embedding_model_loaded': False,
 'controller_training_started': False,
 'recommended_route': 'G2'}

## 4. PRE-A — D0 vs F2-A

Does an external gate around the frozen conservative action have enough headroom? Exact maximum prefers D0 on ties. Epsilon=0.01 prefers D0 whenever it is within epsilon of the best action.


In [4]:
show_csv('action_values.csv')
if (PREFLIGHT/'oracle_summary.csv').exists():
    oracles = pd.read_csv(PREFLIGHT/'oracle_summary.csv')
    display(oracles[oracles.action_space == 'PRE-A'])


,action,n_queries,f1,exact_match,help,hurt,final_prompt_tokens,total_query_prompt_tokens,total_query_generated_tokens,num_lm_calls,evidence_tokens
0,D0,1746,0.385631,0.227950,0.000000,0.000000,940.544101,940.544101,16.654639,1.0,0.000000
1,F0,1746,0.390024,0.226231,0.216495,0.209049,1085.768614,2026.312715,39.949026,2.0,98.332188
2,F2-A,1746,0.389616,0.231386,0.021764,0.008591,1179.446163,2119.990263,33.200458,2.0,98.332188


,action_space,oracle,n_queries,f1,exact_match,best_fixed_f1,headroom,help,hurt,final_prompt_tokens,total_query_prompt_tokens,total_query_generated_tokens,num_lm_calls,evidence_tokens
0,PRE-A,exact,1746,0.393462,0.233677,0.389616,0.003846,0.021764,0.0,946.764032,968.778351,17.608820,1.024055,2.294387
1,PRE-A,epsilon,1746,0.393449,0.233677,0.389616,0.003833,0.021764,0.0,946.046392,966.129439,17.402062,1.021764,2.053837


## 5. PRE-B — D0 vs F0

How much value could perfect acceptance/rejection of the actual fresh answer change recover? Same D0-first exact/epsilon rule. This is the central action-verification ceiling, not an inference result.


In [5]:
if (PREFLIGHT/'oracle_summary.csv').exists():
    display(oracles[oracles.action_space == 'PRE-B'])


,action_space,oracle,n_queries,f1,exact_match,best_fixed_f1,headroom,help,hurt,final_prompt_tokens,total_query_prompt_tokens,total_query_generated_tokens,num_lm_calls,evidence_tokens
2,PRE-B,exact,1746,0.453880,0.266323,0.390024,0.063856,0.216495,0.0,973.217068,1182.256014,23.571019,1.222222,22.205040
3,PRE-B,epsilon,1746,0.453843,0.266323,0.390024,0.063819,0.216495,0.0,972.325315,1176.536082,23.353379,1.216495,21.591065


## 6. PRE-C — D0 vs F0 vs F2-A

Exact maximum tie order is D0 -> F2-A -> F0. Epsilon preference is also D0 -> F2-A -> F0 among actions within 0.01 of the maximum. Selection favors the conservative action; epsilon F1 need not equal exact F1.


In [6]:
if (PREFLIGHT/'oracle_summary.csv').exists():
    display(oracles[oracles.action_space == 'PRE-C'])
show_csv('oracle_distribution.csv')


,action_space,oracle,n_queries,f1,exact_match,best_fixed_f1,headroom,help,hurt,final_prompt_tokens,total_query_prompt_tokens,total_query_generated_tokens,num_lm_calls,evidence_tokens
4,PRE-C,exact,1746,0.454843,0.266896,0.390024,0.064819,0.221077,0.0,975.954754,1189.313860,23.89748,1.227377,22.717068
5,PRE-C,epsilon,1746,0.454801,0.266896,0.390024,0.064777,0.221077,0.0,974.806415,1182.976518,23.65063,1.221077,22.054410


,action_space,oracle,action,count,rate,help_count,hurt_count,neutral_count
0,PRE-A,exact,D0,1704,0.975945,0,0,1704
1,PRE-A,exact,F2-A,42,0.024055,38,0,4
2,PRE-A,epsilon,D0,1708,0.978236,0,0,1708
3,PRE-A,epsilon,F2-A,38,0.021764,38,0,0
4,PRE-B,exact,D0,1358,0.777778,0,0,1358
5,PRE-B,exact,F0,388,0.222222,378,0,10
6,PRE-B,epsilon,D0,1368,0.783505,0,0,1368
7,PRE-B,epsilon,F0,378,0.216495,378,0,0
8,PRE-C,exact,D0,1349,0.772623,0,0,1349
9,PRE-C,exact,F0,366,0.209622,358,0,8


## 7. F2-A unique-action diagnostic

Seven mutually exclusive exact-best/tie classes, F2-A uniquely best and solely epsilon-eligible fractions, strict D0 preservation and F1 equality, and exact/epsilon oracle increments over D0/F0. A large F2-A preservation rate alone does not establish redundancy; use the saved contribution metrics.


In [7]:
show_csv('three_action_unique_best.csv')
if (PREFLIGHT/'headroom_analysis.json').exists():
    headroom = read_json(PREFLIGHT/'headroom_analysis.json')
    display(headroom['f2_unique_contribution'])
    display(headroom['action_label_balance'])


,category,count,rate
0,D0 uniquely best,15,0.008591
1,F0 uniquely best,366,0.209622
2,F2-A uniquely best,13,0.007446
3,tie involving D0/F0,0,0.000000
4,tie involving D0/F2-A,358,0.205040
5,tie involving F0/F2-A,18,0.010309
6,three-way tie,976,0.558992


{'f2_unique_exact_best_count': 13,
 'f2_unique_exact_best_rate': 0.0074455899198167235,
 'f2_only_epsilon_eligible_count': 9,
 'f2_only_epsilon_eligible_rate': 0.005154639175257732,
 'f2_same_strict_answer_as_d0_rate': 0.9438717067583047,
 'f2_same_case_whitespace_normalized_answer_as_d0_rate': 0.9438717067583047,
 'f2_equal_f1_to_d0_rate': 0.9673539518900344,
 'exact_extra_oracle_f1': 0.0009634178579209784,
 'epsilon_extra_oracle_f1': 0.0009579278590522988}

{'n_queries': 1746,
 'counts': {'HELP': 378, 'NEUTRAL': 1003, 'HURT': 365},
 'rates': {'HELP': 0.21649484536082475,
  'NEUTRAL': 0.5744558991981672,
  'HURT': 0.20904925544100803},
 'accept_target_prevalence': 0.21649484536082475,
 'definition': 'FIT target for THIS F0: delta=F0_F1-D0_F1; ACCEPT iff delta>.01; continuous delta retained. Labels never inference features.'}

## 8. GO/NO-GO / 9. Recommended route

Predeclared screens: G1 epsilon headroom >=0.005; G2 epsilon headroom >=0.005 AND advantage over G1 headroom >=0.005; G3 exact extra oracle value beyond D0/F0 >=0.003. The 0.005 material-advantage screen is descriptive, not a significance test. Unique-best frequency is reported without an outcome-tuned cutoff. Prefer binary G2 if G3 does not pass.


In [8]:
if (PREFLIGHT/'go_no_go.json').exists():
    decision = read_json(PREFLIGHT/'go_no_go.json')
    display({k:v for k,v in decision.items() if k != 'artifact_hashes'})
    print('Recommended research route:', decision['recommended_route'])
show_csv('post_generation_candidate_costs.csv')


{'G1': {'decision': 'NO-GO',
  'epsilon_headroom': 0.0038332437934255337,
  'threshold': 0.005,
  'rule': 'epsilon oracle(D0,F2-A) minus best fixed >= threshold'},
 'G2': {'decision': 'GO',
  'epsilon_headroom': 0.06381925695899743,
  'verifier_advantage': 0.059986013165571894,
  'threshold': 0.005,
  'material_advantage_threshold': 0.005,
  'rule': 'epsilon oracle(D0,F0) headroom >=.005 AND advantage over PRE-A headroom >=.005; descriptive screen, not significance'},
 'G3': {'decision': 'NO-GO',
  'extra_exact_oracle_f1': 0.0009634178579209784,
  'unique_best_rate': 0.0074455899198167235,
  'threshold': 0.003,
  'rule': 'Three-action exact oracle minus D0/F0 exact oracle >=.003. Unique-best fraction is descriptive; no outcome-tuned extra cutoff.'},
 'recommended_route': 'G2',
 'controller_training_started': False,
 'lm_calls': 0,
 'interpretation': 'Retrospective action-space research recommendation, not evidence of a learnable/deployable verifier.'}

Recommended research route: G2


,action_space,note,total_query_prompt_tokens,total_query_generated_tokens,num_lm_calls
0,PRE-A,Generate all non-D0 candidates before action v...,2119.990263,33.200458,2.0
1,PRE-B,Generate all non-D0 candidates before action v...,2026.312715,39.949026,2.0
2,PRE-C,Generate all non-D0 candidates before action v...,3205.758877,56.494845,3.0


## 10. Disabled G2 controller scaffolding

The verifier observes question, y0, exact evidence E and the specific F0 proposal y1. It predicts KEEP_D0 vs ACCEPT_F0. Target delta=F0 F1−D0 F1; ACCEPT iff delta>0.01; continuous delta and HELP/NEUTRAL/HURT labels remain FIT-only supervised data. The feature matrix excludes outcomes, oracle labels, paper IDs, gold answers/evidence and split information.

V0: answer edits/lengths/similarity; V1: lexical evidence/change overlap; V2: saved generation lengths only (confidence unavailable); V3: saved E3 cosines/margins/length/source ranks; V4: generic question form/length. Optional MiniLM q/answer/evidence similarities are not computed here. No new embedding model or confidence-generation backfill.

Two fixed lightweight classifiers are scaffolded: logistic regression and small histogram gradient-boosted trees. No LLM verifier, regression proliferation, three-way controller, dynamic k or cascade is implemented.

Paper-separated protocol: five outer FIT paper folds. Four inner paper folds within each outer training fold select the threshold from the preregistered grid using downstream F1. Within 0.003 prefer lower HURT, lower ACCEPT, then cost and simpler threshold. Imputer/scaler fitting is confined to training folds. Outer-heldout labels never select their threshold. A later deployment threshold is chosen from all outer OOF probabilities and explicitly marked post-selection.


In [9]:
scaffold = RESULT_ROOT/'scaffold'
if (scaffold/'feature_schema.json').exists():
    display(read_json(scaffold/'feature_schema.json'))
    display(pd.read_csv(scaffold/'fold_assignments.csv', dtype={'question_id':str,'paper_id':str}).head())
else:
    print('Scaffold code is available; saved feature/fold inspection not created yet.')


{'feature_columns': ['normalized_equal',
  'token_edit_distance',
  'token_jaccard',
  'added_token_count',
  'removed_token_count',
  'initial_answer_length',
  'candidate_answer_length',
  'answer_length_difference',
  'answer_length_ratio',
  'changed_fraction',
  'one_answer_substring',
  'initial_evidence_overlap',
  'candidate_evidence_overlap',
  'evidence_overlap_delta',
  'added_tokens_in_evidence_fraction',
  'removed_tokens_absent_evidence_fraction',
  'initial_generated_tokens',
  'candidate_generated_tokens',
  'top_sentence_cosine',
  'second_sentence_cosine',
  'third_sentence_cosine',
  'mean_selected_cosine',
  'min_selected_cosine',
  'next_unselected_cosine',
  'selected_vs_next_margin',
  'evidence_tokens',
  'mean_source_chunk_rank',
  'selected_chunk_count',
  'query_tokens',
  'question_mark',
  'starts_wh_question',
  'starts_auxiliary_question'],
 'groups': {'V0': ['normalized_equal',
   'token_edit_distance',
   'token_jaccard',
   'added_token_count',
   'rem

,question_id,paper_id,outer_fold
0,10,1,0
1,100,26,1
2,1006,295,1
3,1007,295,1
4,101,26,1


## 11. Optional controller training — OFF

Set `RUN_CONTROLLER_TRAINING=True` only after reviewing the matching preflight decision. The runner requires explicit `--enable-training` and G2 GO. No LM generation occurs even during later training: candidates remain saved. Main metrics are selected-answer F1/EM, HELP/HURT, harm rejection and HELP retention, not AUC alone. All KEEP decisions still pay the already-generated F0 candidate's cost.


In [10]:
if RUN_CONTROLLER_TRAINING:
    subprocess.run([str(ROOT/'sara_env/bin/python'), '-m', 'src.week9.action_verification_g.runner',
                    '--config', str(CONFIG), '--stage', 'train', '--enable-training', '--resume'], cwd=ROOT, check=True)
else:
    print('CONTROLLER TRAINING WAS NOT STARTED')


Validating saved FIT F/E3 actions and hashes; no generator/encoder is loaded.


## Interpretation limits

The preflight quantifies the maximum value among already-generated answers. It cannot establish that the verifier can predict the beneficial action. F0 must exist before post-generation verification, so KEEP does not save the second LM call. Oracle-selected costs are retrospective and exclude searching all actions. Measured end-to-end latency is unavailable with cached upstream work.

FIT was already used to choose E3/F prompts. Future paper OOF evaluates a verifier conditionally on frozen upstream decisions; it is not independent end-to-end confirmation. Do not tune on DEV/INTERNAL-VAL/TEST. No claim of causal evidence, general RAG superiority or controller success is supported by oracle headroom alone.
